In [2]:
import json
from typing import Dict, Any, List

In [3]:
# --- Step 1: The Standardized MCP Tool Server ---

class MCPGitHubServer:
    """
    An isolated, standalone MCP Server. 
    It communicates exclusively using universal protocol schemas.
    """
    def __init__(self):
        self.server_name = "Enterprise-GitHub-MCP-Server"

    def handle_protocol_request(self, message_json: str) -> str:
        """The universal entry point interface parsing standard protocol actions."""
        request = json.loads(message_json)
        method = request.get("method")
        
        # Implementation of standard MCP Lifecycle methods
        if method == "tools/list":
            return json.dumps({
                "status": "SUCCESS",
                "tools": [
                    {
                        "name": "create_repository_issue",
                        "description": "Opens a structural tracking issue inside a specific code repository.",
                        "parameters": {"repo_name": "string", "title": "string"}
                    }
                ]
            })
            
        elif method == "tools/call":
            params = request.get("params", {})
            tool_name = params.get("name")
            arguments = params.get("arguments", {})
            
            if tool_name == "create_repository_issue":
                # Execute the real underlying business utility logic
                execution_result = self._core_create_issue_logic(**arguments)
                return json.dumps({
                    "status": "SUCCESS",
                    "result": execution_result
                })
                
        return json.dumps({"status": "ERROR", "error_message": "Unsupported protocol method call."})

    def _core_create_issue_logic(self, repo_name: str, title: str) -> Dict[str, Any]:
        """Internal private tool code resolving the actual network task."""
        print(f"\n[MCP SERVER EXECUTION] Making authenticated API call to GitHub api.github.com...")
        print(f"  → Opened Issue: '{title}' in repository: '{repo_name}'")
        return {"issue_id": 9942, "status": "OPENED", "assigned_to": "aryan-yadav"}

In [4]:
# --- Step 2: The Plug-and-Play MCP Client Application ---

class ModernAIApplicationClient:
    """
    An AI application acting as an MCP Client. It can plug into ANY server 
    built on the standard protocol without custom adaptation logic.
    """
    def __init__(self, target_mcp_server: MCPGitHubServer):
        # Establish connection channel straight to our target server module
        self.server_connection = target_mcp_server
        print("[MCP CLIENT] Connected to external server channel over standardized interface.")

    def discover_capabilities(self) -> List[Dict[str, Any]]:
        """Queries the server for available capabilities using standard schema formats."""
        protocol_message = json.dumps({"method": "tools/list"})
        response_str = self.server_connection.handle_protocol_request(protocol_message)
        response = json.loads(response_str)
        return response.get("tools", [])

    def trigger_tool_execution(self, name: str, arguments: Dict[str, Any]) -> Dict[str, Any]:
        """Dispatches an execution request to the decoupled tool server."""
        protocol_message = json.dumps({
            "method": "tools/call",
            "params": {
                "name": name,
                "arguments": arguments
            }
        })
        response_str = self.server_connection.handle_protocol_request(protocol_message)
        return json.loads(response_str)

In [5]:
# --- Step 3: Run the Plug-and-Play Integration ---

# 1. Spin up our independent tool server process
independent_server = MCPGitHubServer()

print("--- Scenario A: Plugging Server into Chatbot App ---")
# 2. Connect the server directly into Client App A (e.g., a Support Chatbot UI)
chatbot_client = ModernAIApplicationClient(target_mcp_server=independent_server)

# Discover what tools exist dynamically without hardcoding them in the client app
discovered_tools = chatbot_client.discover_capabilities()
print(f"[Client Discovery] Found available tools from server: {discovered_tools[0]['name']}")

# 3. Simulate an LLM decision loop resolving a user task via standard protocol structures
print("\n--- Scenario B: Execution Handoff Sequence ---")
tool_to_run = discovered_tools[0]['name']
simulated_llm_arguments = {"repo_name": "CampusGrid-Core", "title": "Fix buffer overflow in stream layer"}

mcp_response = chatbot_client.trigger_tool_execution(
    name=tool_to_run, 
    arguments=simulated_llm_arguments
)

print("\n" + "="*60)
print("FINAL DECOUPLED MCP SERVER RESPONSE OVER JSON-RPC STREAMS:")
print("="*60)
print(json.dumps(mcp_response, indent=2))

--- Scenario A: Plugging Server into Chatbot App ---
[MCP CLIENT] Connected to external server channel over standardized interface.
[Client Discovery] Found available tools from server: create_repository_issue

--- Scenario B: Execution Handoff Sequence ---

[MCP SERVER EXECUTION] Making authenticated API call to GitHub api.github.com...
  → Opened Issue: 'Fix buffer overflow in stream layer' in repository: 'CampusGrid-Core'

FINAL DECOUPLED MCP SERVER RESPONSE OVER JSON-RPC STREAMS:
{
  "status": "SUCCESS",
  "result": {
    "issue_id": 9942,
    "status": "OPENED",
    "assigned_to": "aryan-yadav"
  }
}
